In [ ]:
import os
import random
from PIL import Image
from datasets import load_from_disk, Dataset
from torch.utils.data import Dataset as TorchDataset, DataLoader
from transformers import CLIPProcessor
import torch
from sklearn.model_selection import train_test_split

# Prevents parallelism issues with tokenizers
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Path to the merged dataset and image directory
merged_dataset_path = '/home/ubuntu/large_disk/project/Hallucination-Mitigation/data/dataset/merged_filtered_cc12m_with_hallucinations_merged_set'
image_dir = '/home/ubuntu/large_disk/project/Hallucination-Mitigation/data/cc12m-data/cc12m-data'

# Load the dataset from disk
merged_dataset = load_from_disk(merged_dataset_path)

# CLIP processor for image and text tokenization
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

'''
# Step 1: Split the dataset into train, validation, and test sets
# Define the split ratios
train_size = 0.8
val_size = 0.1
test_size = 0.1

# Shuffle and split the dataset
# Convert the dataset into a pandas dataframe to use with train_test_split
dataset_df = merged_dataset.to_pandas()

train_data, temp_data = train_test_split(dataset_df, test_size=(val_size + test_size), random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=test_size / (val_size + test_size), random_state=42)

# Convert pandas dataframes to dictionaries and then to Dataset
train_dataset = Dataset.from_dict(train_data.to_dict(orient="list"))
val_dataset = Dataset.from_dict(val_data.to_dict(orient="list"))
test_dataset = Dataset.from_dict(test_data.to_dict(orient="list"))

# Step 2: Store the datasets for later reloading
train_dataset_path = '/home/ubuntu/large_disk/project/Hallucination-Mitigation/data/dataset/train_dataset'
val_dataset_path = '/home/ubuntu/large_disk/project/Hallucination-Mitigation/data/dataset/val_dataset'
test_dataset_path = '/home/ubuntu/large_disk/project/Hallucination-Mitigation/data/dataset/test_dataset'

train_dataset.save_to_disk(train_dataset_path)
val_dataset.save_to_disk(val_dataset_path)
test_dataset.save_to_disk(test_dataset_path)

'''
train_dataset_path = '/home/ubuntu/large_disk/project/Hallucination-Mitigation/data/dataset/train_dataset'
val_dataset_path = '/home/ubuntu/large_disk/project/Hallucination-Mitigation/data/dataset/val_dataset'
test_dataset_path = '/home/ubuntu/large_disk/project/Hallucination-Mitigation/data/dataset/test_dataset'

train_dataset = load_from_disk(train_dataset_path)
val_dataset = load_from_disk(val_dataset_path)
test_dataset = load_from_disk(test_dataset_path)



print(f"Datasets saved: Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")


In [ ]:
# Function to find and print duplicated keys
from collections import Counter
def print_duplicated_keys(dataset, key_field='key'):
    """
    Identifies and prints duplicated keys in the dataset.

    Args:
        dataset (datasets.Dataset): The loaded dataset.
        key_field (str): The field name to check for duplicates. Defaults to 'key'.
    """
    # Extract all keys
    keys = []
    for sample in dataset:
        if key_field in sample:
            keys.append(sample[key_field])
        else:
            print(f"Warning: '{key_field}' not found in a sample.")
            return

    # Count occurrences of each key
    key_counts = Counter(keys)

    # Identify keys with more than one occurrence
    duplicated_keys = [key for key, count in key_counts.items() if count > 1]

    if duplicated_keys:
        print(f"Duplicated keys ({len(duplicated_keys)} found):")
        for key in duplicated_keys:
            print(key)
    else:
        print("No duplicated keys found.")

# Call the function to check for duplicated keys
print_duplicated_keys(merged_dataset, key_field='key')

In [ ]:
import torch

if torch.cuda.is_available():
    print("CUDA is available. PyTorch can access the GPU.")
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
    print(f"Current GPU device: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available. PyTorch cannot access the GPU.")


In [ ]:
import re
from torchvision import transforms

data_augmentation_transforms = transforms.Compose([
    transforms.RandomResizedCrop(size=224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    transforms.RandomRotation(degrees=15),
    # Do not convert to tensor here

])

def clean_caption(caption, prefix_patterns):
    """
    Removes the specified prefixes from the caption, strips leading/trailing whitespace,
    and capitalizes the first character.

    Args:
        caption (str): The original caption string.
        prefix_patterns (list of compiled regex patterns): Patterns to identify prefixes.

    Returns:
        str: The cleaned caption.
    """
    # Normalize whitespace (remove newlines, extra spaces)
    caption = re.sub(r'\s+', ' ', caption)  # Replace all whitespace (including newlines) with a single space

    # Remove specified patterns
    for pattern in prefix_patterns:
        caption = re.sub(pattern, '', caption)

    caption = caption.strip()  # Remove leading/trailing whitespace

    # Capitalize the first letter if the caption is not empty
    if caption:
        caption = caption[0].upper() + caption[1:]

    return caption




In [ ]:
from PIL import Image
import os
import random
from torch.utils.data import Dataset



class HallucinationDataset(Dataset):
    def __init__(self, dataset, image_dir, transform=None, loader="train"):
        self.dataset = dataset
        self.image_dir = image_dir
        self.transform = transform
        self.loader = loader
        # Compile regex patterns for prefixes
        # Updated regex patterns to handle more variations
        self.prefix_patterns = [
            re.compile(r'^This image displays[:\s]*', re.IGNORECASE),
            re.compile(r'^This image shows[:\s]*', re.IGNORECASE),
            re.compile(r'^In this image[:\s]*', re.IGNORECASE),
            re.compile(r'^Positive Caption:[:\s]*', re.IGNORECASE),  # New pattern to handle direct occurrences
            re.compile(r'^\s*This image displays:\s*', re.IGNORECASE),  # Match 'This image displays:' with leading/trailing spaces
            re.compile(r'^\s*This image shows:\s*', re.IGNORECASE),     # Match 'This image shows:'
            re.compile(r'^\s*In this image:\s*', re.IGNORECASE),        # Match 'In this image:'
            re.compile(r'^\s*This image displays\s*', re.IGNORECASE),  # Match 'This image displays:' with leading/trailing spaces

        ]

        self.hallucination_keys = [
            'Object Hallucination Only',
            'Attribute Hallucination Only',
            'Relation Hallucination Only',
            # 'Object Hallucination Variation',
            # 'Attribute Hallucination Variation',
            # 'Relation Hallucination Variation'
        ]
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        # Fetch the sample
        sample = self.dataset[idx]
        
        # Get the image path using the 'key' field
        image_key = sample['key'] + '.jpg'
        image_path = os.path.join(self.image_dir, image_key)
        
        # Handle missing images
        if not os.path.exists(image_path):
            print(f"Image {image_key} is missing, skipping this sample.")
            return None
        
        try:
            # Load the image without transformation, let CLIPProcessor handle it later
            image = Image.open(image_path)
        except Exception as e:
            print(f"Error loading image {image_key}: {e}, skipping this sample.")
            return None

        # Apply data augmentation
        if self.transform:
            image = self.transform(image)
            
        # Get the positive caption ('vlm_caption')
        positive_caption = sample.get('vlm_caption', None)
        if not positive_caption:
            print(f"Positive caption is missing for sample {sample['uid']}, skipping this sample.")
            return None
        # Clean the positive caption
        positive_caption = clean_caption(positive_caption, self.prefix_patterns)
        # Randomly select one of the hallucinated captions
        available_hallucinations = [
            key for key in self.hallucination_keys 
            if sample.get(key, None) and sample[key] != ''
        ]
        if not available_hallucinations:
            # print(f"No hallucinated captions available for sample {sample['uid']}, skipping this sample.")
            # print(sample['Object Hallucination Only'], sample['Attribute Hallucination Only'], sample['Relation Hallucination Only'], sample['Object Hallucination Variation'], sample['Attribute Hallucination Variation'], sample['Relation Hallucination Variation'])
            return None
        if self.loader == "train" or self.loader == "val":
            hallucination_key = random.choice(available_hallucinations)
        elif self.loader == "Object Hallucination Only":
            hallucination_key = "Object Hallucination Only"
        elif self.loader == "Attribute Hallucination Only":
            hallucination_key = "Attribute Hallucination Only"
        elif self.loader == "Relation Hallucination Only":
            hallucination_key = "Relation Hallucination Only"
            

        # hallucination_key = random.choice(available_hallucinations)
        try:
            negative_caption = sample[hallucination_key]
        except:
            # hallucination_key = random.choice(available_hallucinations)
            # negative_caption = sample[hallucination_key]
            negative_caption = None
        if self.loader == "Positive":
            negative_caption = positive_caption

        
        return image, positive_caption, negative_caption


In [ ]:
def collate_fn(batch):
    # Filter out any None samples
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    
    images, positive_captions, negative_captions = zip(*batch)

    # Process images using CLIPProcessor
    image_inputs = processor(images=list(images), return_tensors="pt", padding=True)
    
    # Process text captions using CLIPProcessor
    positive_inputs = processor(text=list(positive_captions), return_tensors="pt", padding=True, truncation=True)
    negative_inputs = processor(text=list(negative_captions), return_tensors="pt", padding=True, truncation=True)
    
    return (
        image_inputs.pixel_values,
        positive_inputs,      # {'input_ids':..., 'attention_mask':...}
        negative_inputs,      # {'input_ids':..., 'attention_mask':...}
        positive_captions,
        negative_captions,
        images
    )


In [ ]:
# Initialize the dataset
class FilteredDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
    
    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        if sample is not None:
            return sample
        else:
            return None

# Wrap the dataset class with a filter to skip None samples


train_dataset_raw = HallucinationDataset(
    dataset=train_dataset, 
    image_dir=image_dir,
    transform=data_augmentation_transforms,
    loader = "train"
)

filtered_train_dataset_raw = FilteredDataset(train_dataset_raw)

val_dataset_raw = HallucinationDataset(
    dataset=val_dataset, 
    image_dir=image_dir,
    transform=None,
    loader = "val"
)

filtered_val_dataset_raw = FilteredDataset(val_dataset_raw)

test_dataset_raw = HallucinationDataset(
    dataset=test_dataset, 
    image_dir=image_dir,
    transform=None,
    loader = "Object Hallucination Only"
)

filtered_test_dataset_raw = FilteredDataset(test_dataset_raw)


# Create DataLoaders
train_loader = DataLoader(filtered_train_dataset_raw, batch_size=40, shuffle=True, collate_fn=collate_fn, num_workers=4)
val_loader = DataLoader(filtered_val_dataset_raw, batch_size=40, shuffle=False, collate_fn=collate_fn, num_workers=4)
test_loader = DataLoader(filtered_test_dataset_raw, batch_size=40, shuffle=False, collate_fn=collate_fn, num_workers=4)



# # DataLoader with filtering for None samples
# dataloader = DataLoader(filtered_dataset, batch_size=40, shuffle=True, collate_fn=collate_fn)

In [ ]:
import matplotlib.pyplot as plt
import torch

def check_samples(dataloader, num_samples=5):
    samples_checked = 0
    for batch in dataloader:
        if batch is None:
            continue
        
        images, positives, negatives, positive_captions, negative_captions, _ = batch
        
        for i in range(len(images)):
            # Convert the image tensor back to PIL for visualization
            image = images[i].permute(1, 2, 0).cpu().numpy()  # Convert to HWC format for plotting
            
            # Plot the image
            plt.figure(figsize=(3, 3))
            plt.imshow(image)
            plt.axis('off')
            plt.title(f"Sample {samples_checked + 1}")
            plt.show()

            # Print positive and negative captions
            print(f"Positive Caption: {positive_captions[i]}")
            print(f"Negative Caption: {negative_captions[i]}")
            print("-" * 50)
            
            samples_checked += 1
            if samples_checked >= num_samples:
                return

# Call the function after creating the dataloader
check_samples(train_loader, num_samples=5)


In [ ]:
# Modified code for CustomCLIPModel to interact with all transformer layers in CLIP

import torch
import torch.nn as nn
from transformers import CLIPModel, CLIPProcessor

class CustomCLIPModel(nn.Module):
    def __init__(self, clip_model_name="openai/clip-vit-base-patch32"):
        super(CustomCLIPModel, self).__init__()
        
        # Load the pre-trained CLIP model with hidden states enabled
        self.clip_model = CLIPModel.from_pretrained(clip_model_name, output_hidden_states=True)
        
        # Freeze the CLIP model parameters (optional)
        for param in self.clip_model.parameters():
            param.requires_grad = True
            
        # Define the query tokens for Q-Former (learnable parameters)
        self.num_query_tokens = 1
        self.query_tokens = nn.Parameter(torch.randn(1, self.num_query_tokens, self.clip_model.config.projection_dim))
        
        # Define a cross-attention layer to interact with each transformer layer in the vision and text encoders
        self.cross_attention = nn.MultiheadAttention(self.clip_model.config.projection_dim, num_heads=8)

        # Define the binary classification subnetwork
        self.binary_classifier = nn.Sequential(
            nn.Linear(self.clip_model.config.projection_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )


        # Projection layers for each block, excluding the last one
        self.projection_layers = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(self.clip_model.vision_model.config.hidden_size),
                nn.Linear(self.clip_model.vision_model.config.hidden_size, self.clip_model.config.projection_dim)
            ) for _ in range(len(self.clip_model.vision_model.encoder.layers) - 1)
        ])
        # print("the layers number:", self.clip_model.vision_model.encoder.layers)


    def extract_text_embeds(self, text_inputs):
        """ Extracts the text embeddings using the CLIP text model. """
        text_output = self.clip_model.text_model(
            input_ids=text_inputs['input_ids'], 
            attention_mask=text_inputs['attention_mask'],
            output_hidden_states=True
        )
        text_hidden_states = text_output.hidden_states
        if text_hidden_states is None:
            raise ValueError("Hidden states for text embeddings are not available.")
        return text_hidden_states

    def extract_image_embeds(self, image_inputs):
        """ Extracts the image embeddings using the CLIP vision model. """
        image_output = self.clip_model.vision_model(
            pixel_values=image_inputs,
            output_hidden_states=True
        )
        image_hidden_states = image_output.hidden_states
        if image_hidden_states is None:
            raise ValueError("Hidden states for image embeddings are not available.")
        return image_hidden_states

    def forward(self, image_inputs, text_inputs_positive, text_inputs_negative):
        # Extract image and text embeddings (full token-level embeddings from all layers)
        image_hidden_states = self.extract_image_embeds(image_inputs)  # List of hidden states from each layer
        text_hidden_states_positive = self.extract_text_embeds(text_inputs_positive)
        text_hidden_states_negative = self.extract_text_embeds(text_inputs_negative)
        
        query_tokens_pos = self.query_tokens.expand(image_inputs.shape[0], -1, -1)
        query_tokens_neg = query_tokens_pos.clone()  # Negative query is a duplicate of the positive query

        # Loop over all transformer layers and apply cross-attention
        # print("the length of the image_hidden_states", len(image_hidden_states))
        for i in range(1, len(image_hidden_states)):
            # Get the hidden states from the i-th transformer layer
            image_features = image_hidden_states[i]
            text_features_pos = text_hidden_states_positive[i]
            text_features_neg = text_hidden_states_negative[i]
            
            # Project image features to match the projection dimension if needed
            if i == (len(image_hidden_states) - 1):
                # print("i idx number for the last", i)
                image_features = self.clip_model.vision_model.post_layernorm(image_features)
                image_features = self.clip_model.visual_projection(image_features)
            else:
                # print("i idx number", i)
                image_features = self.projection_layers[i -1](image_features)

            # Combine image and text features
            combined_features_pos = torch.cat([image_features, text_features_pos], dim=1)
            combined_features_neg = torch.cat([image_features, text_features_neg], dim=1)
            
            # Apply cross-attention for positive captions
            combined_features_pos = combined_features_pos.permute(1, 0, 2)  # Shape: [seq_len, batch_size, projection_dim]
            query_tokens_pos = query_tokens_pos.permute(1, 0, 2)  # Shape: [num_query_tokens, batch_size, projection_dim]
            query_tokens_pos, _ = self.cross_attention(query_tokens_pos, combined_features_pos, combined_features_pos)
            query_tokens_pos = query_tokens_pos.permute(1, 0, 2)  # Shape: [batch_size, num_query_tokens, projection_dim]
            
            # Apply cross-attention for negative captions
            combined_features_neg = combined_features_neg.permute(1, 0, 2)
            query_tokens_neg = query_tokens_neg.permute(1, 0, 2)
            query_tokens_neg, _ = self.cross_attention(query_tokens_neg, combined_features_neg, combined_features_neg)
            query_tokens_neg = query_tokens_neg.permute(1, 0, 2)
        
        # Normalize embeddings
        query_tokens_attended_pos = query_tokens_pos / query_tokens_pos.norm(p=2, dim=-1, keepdim=True)
        query_tokens_attended_neg = query_tokens_neg / query_tokens_neg.norm(p=2, dim=-1, keepdim=True)
        
        # Cosine similarity logits
        logits_query_pos = torch.matmul(query_tokens_attended_pos, image_features[:, 0, :].unsqueeze(-1)).squeeze(-1) 
        logits_query_neg = torch.matmul(query_tokens_attended_neg, image_features[:, 0, :].unsqueeze(-1)).squeeze(-1) 

        # Binary classification logits
        logits_binary_classification_pos = self.binary_classifier(query_tokens_attended_pos.squeeze(0))
        logits_binary_classification_neg = self.binary_classifier(query_tokens_attended_neg.squeeze(0))

        return _, _, logits_binary_classification_pos, logits_binary_classification_neg, logits_query_pos, logits_query_neg
    
    def infer(self, image_inputs, text_inputs):
        # Extract image and text embeddings
        image_hidden_states = self.extract_image_embeds(image_inputs)
        text_hidden_states = self.extract_text_embeds(text_inputs)
        
        query_tokens = self.query_tokens.expand(image_inputs.shape[0], -1, -1)

        # Loop over all transformer layers and apply cross-attention
        for i in range(1, len(image_hidden_states)):
            # Get the hidden states from the i-th transformer layer
            image_features = image_hidden_states[i]
            text_features = text_hidden_states[i]
            
            # Project image features to match the projection dimension if needed
            if i == (len(image_hidden_states) - 1):
                image_features = self.clip_model.vision_model.post_layernorm(image_features)
                image_features = self.clip_model.visual_projection(image_features)
            else:
                image_features = self.projection_layers[i - 1](image_features)

            # Combine image and text features
            combined_features = torch.cat([image_features, text_features], dim=1)
            
            # Apply cross-attention
            combined_features = combined_features.permute(1, 0, 2)  # Shape: [seq_len, batch_size, projection_dim]
            query_tokens = query_tokens.permute(1, 0, 2)  # Shape: [num_query_tokens, batch_size, projection_dim]
            query_tokens, _ = self.cross_attention(query_tokens, combined_features, combined_features)
            query_tokens = query_tokens.permute(1, 0, 2)  # Shape: [batch_size, num_query_tokens, projection_dim]
        
        # Normalize embeddings
        query_tokens_attended = query_tokens / query_tokens.norm(p=2, dim=-1, keepdim=True)
        
        # Cosine similarity logits
        logits_query = torch.matmul(query_tokens_attended, image_features[:, 0, :].unsqueeze(-1)).squeeze(-1)

        # Binary classification logits
        logits_binary_classification = self.binary_classifier(query_tokens_attended.squeeze(0))

        return logits_query, logits_binary_classification


In [ ]:
import torch
from PIL import Image
from transformers import CLIPProcessor

# Load the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CustomCLIPModel()
model.to(device)

# Load a test image
# Dummy image and text inputs
image_path = "/home/ubuntu/large_disk/project/Hallucination-Mitigation/data/dataset/000510011.jpg"
image = Image.open(image_path)
text_input_positive = "A group of people standing in front of a building."
text_input_negative = "A flying car above the mountains."

# Load the CLIP processor for tokenization and image preprocessing
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Preprocess the image and text
inputs = processor(text=[text_input_positive, text_input_negative], images=image, return_tensors="pt", padding=True)

# Move inputs to device
image_inputs = inputs['pixel_values'].to(device)
text_inputs_positive = {
    'input_ids': inputs['input_ids'][0].unsqueeze(0).to(device),
    'attention_mask': inputs['attention_mask'][0].unsqueeze(0).to(device)
}
text_inputs_negative = {
    'input_ids': inputs['input_ids'][1].unsqueeze(0).to(device),
    'attention_mask': inputs['attention_mask'][1].unsqueeze(0).to(device)
}

# Forward pass through the model
model.eval()
with torch.no_grad():
     _, _, logits_binary_classification_pos, logits_binary_classification_neg, logits_query_pos, logits_query_neg= model(
        image_inputs, text_inputs_positive, text_inputs_negative
    )

# Print the results
print("Cosine Similarity Logits (Positive Caption + Image):", logits_query_pos)
print("Cosine Similarity Logits (Negative Caption + Image):", logits_query_neg)
print("Binary Classification Logits (Positive Caption + Image):", logits_binary_classification_pos)
print("Binary Classification Logits (Negative Caption + Image):", logits_binary_classification_neg)


In [ ]:
1

In [ ]:
import torch.nn.functional as F

class CosineTripletLoss(nn.Module):
    def __init__(self, margin=0.5):
        super(CosineTripletLoss, self).__init__()
        self.margin = margin
    
    def forward(self, similarity_pos, similarity_neg):
        """
        Args:
            similarity_pos: Cosine similarity between image and positive text (batch_size,)
            similarity_neg: Cosine similarity between image and negative text (batch_size,)
        Returns:
            Triplet loss value
        """
        # Calculate the loss: max( similarity_neg - similarity_pos + margin, 0 )
        loss = F.relu(similarity_neg - similarity_pos + self.margin)
        return loss.mean()

criterion_binary = nn.BCELoss()


In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, delta=0, save_path='best_model.pth'):
        self.patience = patience
        self.delta = delta
        self.save_path = save_path
        self.counter = 0
        self.best_score = None
        self.early_stop = False
    
    def __call__(self, val_loss, model):
        score = -val_loss  # Assuming we want to minimize validation loss
        
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0
    
    def save_checkpoint(self, val_loss, model):
        '''Saves model when validation loss decreases.'''
        torch.save(model.state_dict(), self.save_path)
        print(f'Validation loss decreased, saving model to {self.save_path}')

def validate_model(model, val_loader, triplet_loss_fn, bce_loss_fn, device):
    model.eval()
    running_loss = 0.0
    running_triplet_loss = 0.0
    running_bce_loss = 0.0
    running_triplet_loss_query = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for batch in val_loader:
            if batch is None:
                continue
            
            image_inputs, positive_text_inputs, negative_text_inputs, _, _, _ = batch
            
            # Move inputs to device
            image_inputs = image_inputs.to(device)
            positive_text_inputs = {k: v.to(device) for k, v in positive_text_inputs.items()}
            negative_text_inputs = {k: v.to(device) for k, v in negative_text_inputs.items()}
            
            # Forward pass
            _, _, logits_binary_pos, logits_binary_neg, similarity_query_pos, similarity_query_neg= model(
                image_inputs, positive_text_inputs, negative_text_inputs
            )
            
            # Compute losses
            # triplet_loss = triplet_loss_fn(similarity_pos, similarity_neg)
            triplet_loss_query = triplet_loss_fn(similarity_query_pos, similarity_query_neg)

            labels_pos = torch.ones_like(logits_binary_pos).to(device)
            labels_neg = torch.zeros_like(logits_binary_neg).to(device)
            bce_loss_pos = bce_loss_fn(logits_binary_pos, labels_pos)
            bce_loss_neg = bce_loss_fn(logits_binary_neg, labels_neg)
            bce_loss = bce_loss_pos + bce_loss_neg  # Total BCE loss
            total_loss =  bce_loss + triplet_loss_query  # Total loss
            
            # Accumulate losses
            # running_triplet_loss += triplet_loss.item()
            running_triplet_loss_query += triplet_loss_query.item()
            running_bce_loss += bce_loss.item()
            running_loss += total_loss.item()
            num_batches += 1
        
    # Calculate average losses
    # avg_triplet_loss = running_triplet_loss / num_batches
    avg_triplet_loss_query = running_triplet_loss_query / num_batches
    avg_bce_loss = running_bce_loss / num_batches
    avg_loss = running_loss / num_batches
    return avg_loss, _, avg_bce_loss, avg_triplet_loss_query



In [ ]:
from torch.optim.lr_scheduler import StepLR
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm


writer = SummaryWriter(log_dir='./runs/experiment_with_variations')


def train_model(model, train_loader, val_loader, num_epochs=10, lr=1e-5, device='cuda'):
    model.to(device)
    
    # Initialize loss functions
    triplet_loss_fn = CosineTripletLoss(margin=0.5)
    bce_loss_fn = nn.BCELoss()
    
    # Define optimizer (only parameters that require gradients)
    # optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = StepLR(optimizer, step_size=5, gamma=0.1)  # Example: Reduce LR every 5 epochs





    # Early Stopping
    early_stopping = EarlyStopping(patience=3, save_path='./10_OCT_Prompt_Model_Full_learning_best_model.pth')
    
    for epoch in range(num_epochs):
        model.train()
        running_triplet_loss = 0.0
        running_bce_loss = 0.0
        running_total_loss = 0.0
        running_triplet_loss_query = 0.0
        num_batches = 0
        progress_bar = tqdm(
            enumerate(train_loader),
            total=len(train_loader),
            desc=f"Epoch {epoch+1}/{num_epochs}"
        )
        
        for batch_idx, batch in progress_bar:
            if batch is None:
                continue

            
            image_inputs, positive_text_inputs, negative_text_inputs, _, _, _ = batch
            
            # Move inputs to the device
            image_inputs = image_inputs.to(device)
            positive_text_inputs = {k: v.to(device) for k, v in positive_text_inputs.items()}
            negative_text_inputs = {k: v.to(device) for k, v in negative_text_inputs.items()}
            
            # Forward pass
            _, _, logits_binary_pos, logits_binary_neg, similarity_pos_query, similarity_neg_query = model(
                image_inputs, positive_text_inputs, negative_text_inputs
            )
            
            # Triplet Loss
            # triplet_loss = triplet_loss_fn(similarity_pos, similarity_neg)
            triplet_loss_query = triplet_loss_fn(similarity_pos_query, similarity_neg_query)
            
            # Binary Classification Loss
            # Positive captions: label=1
            # Negative captions: label=0
            labels_pos = torch.ones_like(logits_binary_pos).to(device)
            labels_neg = torch.zeros_like(logits_binary_neg).to(device)
            
            bce_loss_pos = bce_loss_fn(logits_binary_pos, labels_pos)
            bce_loss_neg = bce_loss_fn(logits_binary_neg, labels_neg)
            bce_loss = bce_loss_pos + bce_loss_neg  # Total BCE loss
            
            # Total Loss: Weighted sum (you can adjust weights if necessary)
            total_loss =   bce_loss + triplet_loss_query

            # Backward and optimize
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            # Accumulate losses for logging
            # running_triplet_loss += triplet_loss.item()
            running_triplet_loss_query += triplet_loss_query.item()
            running_bce_loss += bce_loss.item()
            running_total_loss += total_loss.item()
            num_batches += 1

                        # Update progress bar with current loss
            progress_bar.set_postfix({
                'Batch': batch_idx + 1,
                # 'Triplet Loss': f"{triplet_loss.item():.4f}",
                'Triplet Loss Query': f"{triplet_loss_query.item():.4f}",
                'BCE Loss': f"{bce_loss.item():.4f}",
                'Total Loss': f"{total_loss.item():.4f}"
            })

            # Log batch-level losses to TensorBoard
            global_step = epoch * len(train_loader) + batch_idx
            # writer.add_scalar('Loss/Train_Triplet', triplet_loss.item(), global_step)
            writer.add_scalar('Loss/Train_Triplet_Query', triplet_loss_query.item(), global_step)
            writer.add_scalar('Loss/Train_BCE', bce_loss.item(), global_step)
            writer.add_scalar('Loss/Train_Total', total_loss.item(), global_step)

        
        # avg_triplet_loss = running_triplet_loss / num_batches
        avg_triplet_loss_query = running_triplet_loss_query / num_batches
        avg_bce_loss = running_bce_loss / num_batches
        avg_total_loss = running_total_loss / num_batches

        # Validation Phase
        val_loss, _, val_bce_loss, avg_triplet_loss_query= validate_model(model, val_loader, triplet_loss_fn, bce_loss_fn, device)
            
        print(f"Epoch [{epoch+1}/{num_epochs}],, Triplet Loss Query: {avg_triplet_loss_query:.4f}  ,BCE Loss: {avg_bce_loss:.4f}, Total Loss: {avg_total_loss:.4f}, Val BCE Loss: {val_bce_loss}, Val Loss: {val_loss:.4f}")
        scheduler.step()
        # At the end of each epoch
        writer.add_scalar('Loss/Train', avg_total_loss, epoch)
        writer.add_scalar('Learning Rate', optimizer.param_groups[0]['lr'], epoch)
        # Inside your training loop, after validation
        writer.add_scalar('Loss/Validation_Total', val_loss, epoch)
        # writer.add_scalar('Loss/Validation_Triplet', val_triplet_loss, epoch)
        writer.add_scalar('Loss/Validation_BCE', val_bce_loss, epoch)
            
        # Early Stopping Check
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print("Early stopping triggered.")
            break


    print("Training complete!")
    writer.close()

# torch.save(model.state_dict(), 'full_model_trained_final.pth')


In [ ]:
train_model(model, train_loader, val_loader, num_epochs=50, lr=1e-5, device=device)


In [ ]:
# Load model weights
def load_model_weights(model, weights_path):
    state_dict = torch.load(weights_path)
    model.load_state_dict(state_dict, strict=False)

# Example usage
model = CustomCLIPModel()
load_model_weights(model, "/home/ubuntu/large_disk/project/Hallucination-Mitigation/best_model/10_OCT_Prompt_Model_Full_learning_best_model.pth")

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import numpy as np
from tqdm import tqdm

def evaluate_model_batch(model, dataloader, device='cuda'):
    """
    Evaluates the model on the given dataloader using batch-level inference.
    
    Args:
        model (CustomCLIPModel): The trained model instance.
        dataloader (DataLoader): DataLoader for evaluation data.
        device (str): Device to perform evaluation on ('cuda' or 'cpu').
    
    Returns:
        None
    """
    model.to(device)
    model.eval()
    
    # Accumulators for combined metrics
    all_labels = []
    all_preds = []
    all_probs = []
    
    # Accumulators for positive sample metrics
    all_labels_pos = []
    all_preds_pos = []
    
    # Accumulators for negative sample metrics
    all_labels_neg = []
    all_preds_neg = []
    
    with torch.no_grad():
        # Initialize progress bar
        progress_bar = tqdm(enumerate(dataloader), total=len(dataloader), desc="Evaluating")
        for batch_idx, batch in progress_bar:
            if batch is None:
                continue
            
            # Unpack the batch
            image_inputs, positive_text_inputs, negative_text_inputs, positive_captions, negative_captions, _ = batch
            
            # Move inputs to device
            image_inputs = image_inputs.to(device)
            positive_text_inputs = {k: v.to(device) for k, v in positive_text_inputs.items()}
            negative_text_inputs = {k: v.to(device) for k, v in negative_text_inputs.items()}
            
            # Process positive samples
            logits_similarity_pos, classifications_pos = model.infer(
                image_inputs=image_inputs,
                text_inputs=positive_text_inputs
            )
            
            # Process negative samples
            logits_similarity_neg, classifications_neg = model.infer(
                image_inputs=image_inputs,
                text_inputs=negative_text_inputs
            )
            
            # Labels for positive and negative samples
            labels_pos = torch.ones_like(classifications_pos, dtype=torch.int)
            labels_neg = torch.zeros_like(classifications_neg, dtype=torch.int)
            
            # Predictions (threshold set to 0.5)
            preds_pos = (classifications_pos > 0.5).int()
            preds_neg = (classifications_neg > 0.5).int()
            
            # Probabilities
            probs_pos = classifications_pos.cpu().numpy()
            probs_neg = classifications_neg.cpu().numpy()
            
            # Combined evaluation
            all_labels.extend(labels_pos.cpu().numpy().flatten())
            all_labels.extend(labels_neg.cpu().numpy().flatten())
            all_preds.extend(preds_pos.cpu().numpy().flatten())
            all_preds.extend(preds_neg.cpu().numpy().flatten())
            all_probs.extend(probs_pos.flatten())
            all_probs.extend(probs_neg.flatten())
            
            # Positive samples evaluation (accuracy only)
            all_labels_pos.extend(labels_pos.cpu().numpy().flatten())
            all_preds_pos.extend(preds_pos.cpu().numpy().flatten())
            
            # Negative samples evaluation (accuracy only)
            all_labels_neg.extend(labels_neg.cpu().numpy().flatten())
            all_preds_neg.extend(preds_neg.cpu().numpy().flatten())
            
    # Convert lists to numpy arrays for compatibility with scikit-learn metrics
    all_labels = np.array(all_labels, dtype=int)
    all_preds = np.array(all_preds, dtype=int)
    all_probs = np.array(all_probs)

    # Calculate epoch-level metrics for combined samples
    print("the length of the all_labels", len(all_labels))
    print("the length of the all_preds", len(all_preds))
    print("the first 10 samples of all_labels", all_labels[:10])
    print("the first 10 samples of all_preds", all_preds[:10])

    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary', zero_division=0
    )
    try:
        roc_auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        roc_auc = np.nan  # Handle cases where ROC-AUC cannot be computed

    # Calculate epoch-level accuracy for positive samples
    all_labels_pos = np.array(all_labels_pos, dtype=int)
    all_preds_pos = np.array(all_preds_pos, dtype=int)
    accuracy_pos = accuracy_score(all_labels_pos, all_preds_pos)

    # Calculate epoch-level accuracy for negative samples
    all_labels_neg = np.array(all_labels_neg, dtype=int)
    all_preds_neg = np.array(all_preds_neg, dtype=int)
    accuracy_neg = accuracy_score(all_labels_neg, all_preds_neg)
    
    # Print final evaluation results
    print(f"\nEpoch Evaluation Results (Combined):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")
    
    print(f"\nEpoch Evaluation Results (Positive Samples Only):")
    print(f"Accuracy: {accuracy_pos:.4f}")
    
    print(f"\nEpoch Evaluation Results (Negative Samples Only):")
    print(f"Accuracy: {accuracy_neg:.4f}")


In [ ]:
# %%
# Initialize the Model
model = CustomCLIPModel()

# Load the Trained Model
# model.load_state_dict(torch.load('/home/ubuntu/large_disk/project/Hallucination-Mitigation/model_oct/10_OCT_Prompt_Model_best_model.pth'))
model.load_state_dict(torch.load('/home/ubuntu/large_disk/project/Hallucination-Mitigation/model_oct/10_OCT_Prompt_Model_Full_learning_best_model.pth'))
model.to(device)
model.eval()

# Define Evaluation DataLoader (if separate from training)
# For demonstration, we'll use the existing `dataloader`

# Perform Evaluation


In [ ]:
# Perform Evaluation
model.eval()

evaluate_model_batch(model, val_loader, device=device)

In [ ]:
# Perform Evaluation
model.eval()



test_dataset_raw = HallucinationDataset(
    dataset=test_dataset, 
    image_dir=image_dir,
    transform=None,
    loader = "Object Hallucination Only"
)

filtered_test_dataset_raw = FilteredDataset(test_dataset_raw)


test_loader = DataLoader(filtered_test_dataset_raw, batch_size=40, shuffle=False, collate_fn=collate_fn, num_workers=4)



evaluate_model_batch(model, test_loader, device=device)

In [ ]:
# Perform Evaluation
model.eval()



test_dataset_raw = HallucinationDataset(
    dataset=test_dataset, 
    image_dir=image_dir,
    transform=None,
    loader = "Attribute Hallucination Only"
)

filtered_test_dataset_raw = FilteredDataset(test_dataset_raw)


test_loader = DataLoader(filtered_test_dataset_raw, batch_size=40, shuffle=False, collate_fn=collate_fn, num_workers=4)



evaluate_model_batch(model, test_loader, device=device)

In [ ]:
# Perform Evaluation
model.eval()



test_dataset_raw = HallucinationDataset(
    dataset=test_dataset, 
    image_dir=image_dir,
    transform=None,
    loader = "Relation Hallucination Only"
)

filtered_test_dataset_raw = FilteredDataset(test_dataset_raw)


test_loader = DataLoader(filtered_test_dataset_raw, batch_size=40, shuffle=False, collate_fn=collate_fn, num_workers=4)



evaluate_model_batch(model, test_loader, device=device)